### PySpark: Handling Corrupted Data & Read Modes Lab

#### Objective
Learn how to handle corrupted/malformed data in PySpark using different read modes:
- **PERMISSIVE** (default): Stores corrupted records in `_corrupt_record` column
- **DROPMALFORMED**: Ignores corrupted records
- **FAILFAST**: Throws an exception when it encounters corrupted data

#### Lab Structure
1. Create sample data with corrupted records
2. Test PERMISSIVE mode
3. Test DROPMALFORMED mode
4. Test FAILFAST mode
5. Test badRecordsPath option for production monitoring

In [0]:
# Create sample CSV data with some corrupted records
import os

# Use workspace path for sample data (accessible on serverless)
data_dir = "/Workspace/Users/dipvirmani28@gmail.com/data-engineering-prep/corrupted_data_folder"
dbutils.fs.mkdirs(data_dir)

# Sample CSV data with intentional corruption:
# - Line 3: Missing age field
# - Line 5: Extra field
# - Line 7: Text in numeric field
sample_data = """id,name,age,city
1,John,30,NewYork
2,Alice,25,LosAngeles
3,Bob,Chicago
4,Charlie,35,Houston
5,David,28,Phoenix,ExtraField
6,Eve,22,Philadelphia
7,Frank,Invalid,SanDiego
8,Grace,29,SanAntonio
9,Henry,31,Dallas
10,Ivy,27,SanJose"""

# Write sample data to file
file_path = f"{data_dir}/people.csv"
with open(f"/Workspace{file_path}", "w") as f:
    f.write(sample_data)

print(f"Sample data created at: {file_path}")
print(f"\nData preview:")
print(sample_data)

In [0]:
# PERMISSIVE mode: Default behavior
# - Stores corrupted records in a special column called '_corrupt_record'
# - Does NOT fail the job

# Define schema
schema = "id INT, name STRING, age INT, city STRING, _corrupt_record STRING"

# Read with PERMISSIVE mode (default)
df_permissive = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .csv(f"{data_dir}/people.csv")

print("=== PERMISSIVE Mode ===")
print(f"Total records read: {df_permissive.count()}")
print("\nAll records (including corrupted):")
display(df_permissive)

In [0]:
# With PERMISSIVE mode, corrupted CSV records are stored in the '_corrupt_record' column
# Identify corrupted records by checking for non-null '_corrupt_record'

from pyspark.sql.functions import col

# Corrupted records: non-null _corrupt_record
corrupted_records = df_permissive.filter(col("_corrupt_record").isNotNull())
display(corrupted_records)

# Clean records: null _corrupt_record
clean_records = df_permissive.filter(col("_corrupt_record").isNull())
display(clean_records)

In [0]:
# DROPMALFORMED mode:
# - Automatically drops/ignores corrupted records
# - Only returns valid records that match the schema
# - Does NOT fail the job

df_drop = spark.read \
    .option("header", "true") \
    .option("mode", "DROPMALFORMED") \
    .schema(schema) \
    .csv(f"{data_dir}/people.csv")

print("=== DROPMALFORMED Mode ===")
print("\nOnly valid records (corrupted records dropped):")
display(df_drop)

print("\nNote: Corrupted records are silently dropped.")
print("Original file had 10 data rows, but only valid ones are shown.")

In [0]:
# FAILFAST mode:
# - Throws an exception immediately when it encounters corrupted data
# - Use this when data quality is critical
# - Job will fail if any record doesn't match the schema

print("=== FAILFAST Mode ===")
print("Attempting to read with FAILFAST mode...")

try:
    df_failfast = spark.read \
        .option("header", "true") \
        .option("mode", "FAILFAST") \
        .schema(schema) \
        .csv(f"{data_dir}/people.csv")
    
    # This will fail before we can display anything
    display(df_failfast)
except Exception as e:
    print(f"\n❌ ERROR: {type(e).__name__}")
    print(f"Message: {str(e)}")
    print("\nFAILFAST mode throws an exception when corrupted data is found.")
    print("This is useful for critical data pipelines where data quality is mandatory.")

In [0]:
# badRecordsPath option:
# - Automatically saves corrupted/malformed records to a separate directory
# - IMPORTANT: Cannot be combined with explicit 'mode' option (uses default PERMISSIVE behavior)
# - Essential for production data quality monitoring
# - Captures the raw corrupted records for debugging

# Set up path for bad records
bad_records_path = f"/Volumes/dbacademy/default/bad_records/"

# Clean up any existing bad records from previous runs
dbutils.fs.rm(bad_records_path, recurse=True)

print("=== badRecordsPath Option ===")
print(f"Bad records will be saved to: {bad_records_path}\n")

# Read with badRecordsPath option (using PERMISSIVE mode)
df_with_bad_path = spark.read \
    .option("header", "true") \
    .option("badRecordsPath", bad_records_path) \
    .schema(schema) \
    .csv(f"{data_dir}/people.csv")

print(f"Total records read: {df_with_bad_path.count()}")
print("\nAll records:")
display(df_with_bad_path)

#### Summary of Read Modes

| Mode | Behavior | Use Case |
|------|----------|----------|
| **PERMISSIVE** | Stores corrupted records with nulls; raw data in `_corrupt_record` | When you want to analyze and fix corrupted data |
| **DROPMALFORMED** | Silently drops corrupted records | When corrupted data is acceptable to ignore |
| **FAILFAST** | Throws exception on first corrupted record | When data quality is critical and corruption should stop the pipeline |
| **badRecordsPath** | Saves corrupted records to a separate directory (cannot be combined with explicit 'mode' option) | Production data quality monitoring and debugging |

#### Best Practices

1. **Use PERMISSIVE for data exploration** - helps identify data quality issues
2. **Use columnNameOfCorruptRecord** - captures the actual corrupted line for debugging
3. **Use DROPMALFORMED for production** - when some data loss is acceptable
4. **Use FAILFAST for critical pipelines** - ensures data quality or alerts on issues
5. **Use badRecordsPath in production** - automatically save corrupted records to a separate directory for analysis
6. **Monitor corruption rates** - track percentage of corrupted records over time